In [1]:
import os
import asyncpg
import pandas as pd
from dotenv import load_dotenv
import json

# Load environment variables from .env file
load_dotenv('../.env.production')

# Get database URL from environment
database_url = os.getenv('DATABASE_URL')

In [3]:

async def fetch_data():
    # Connect to the database
    conn = await asyncpg.connect(database_url)
    
    # Fetch log table data
    records = await conn.fetch('SELECT * FROM log')
    
    # Convert to DataFrame
    if records:
        log_df = pd.DataFrame(records, columns=records[0].keys())
    else:
        log_df = pd.DataFrame()
    
    await conn.close()
    return log_df

# Run the async function
log_df = await fetch_data()

log_df = log_df[log_df.deleted_at.isna()]
log_df["data"] = log_df["data"].apply(lambda x: json.loads(x))

log_df.head()

,id,created_at,updated_at,deleted_at,text,data,version
6,01JM15636T586W3SPBTK9Z5YQ1,2025-02-14T02:39:12.090Z,2025-02-14T02:39:12.090Z,None,had a hot chocolate,"[{'item': 'hot chocolate', 'action': 'drank', ...",80
15,01JM2CTWX8TBXB1NHD4BV9R3KA,2025-02-14T14:12:01.000Z,2025-02-14T14:12:01.000Z,None,Tried to poop but couldn’t,"[{'effort': 'high', 'schema': 'pooped', 'poopT...",88
17,01JM2K7HFWZVCAPJFR75VCE0HS,2025-02-14T16:03:48.000Z,2025-02-14T16:03:48.000Z,None,Stomach feeling a bit unsettled,"[{'schema': 'symptom', 'startedAt': '2025-02-1...",90
21,01JM2M1P2M8BAD8EJ4XFM7NH2Y,2025-02-14T16:18:03.000Z,2025-02-14T16:18:03.000Z,None,Small poop. A little wet,"[{'effort': 'low', 'schema': 'pooped', 'poopTy...",91
22,01JM2VK0XF7Y22PK1M9DJBZPXQ,2025-02-14T18:29:54.000Z,2025-02-14T18:29:54.000Z,None,Vegetable and chickpea soup and 1 glass of water,"[{'item': 'vegetable and chickpea soup', 'acti...",92


In [25]:
events_df = log_df.explode("data").dropna(subset=["data"])
events_df["source_text"] = events_df["text"]
events_df = events_df[["created_at", "data", "source_text"]]
events_df.tail(10)

,created_at,data,source_text
558,2025-04-02T07:26:16-04:00,"{'item': 'vitamin d', 'action': 'took', 'amoun...","Took omega 3, vitamin d, and 5 Metamucil capsules"
558,2025-04-02T07:26:16-04:00,"{'item': 'Metamucil capsules', 'action': 'took...","Took omega 3, vitamin d, and 5 Metamucil capsules"
559,2025-04-02T13:08:25-04:00,"{'item': 'chickpea curry on rice', 'action': '...",Ate chickpea curry on rice and salad. Drank 35...
559,2025-04-02T13:08:25-04:00,"{'item': 'salad', 'action': 'ate', 'amount': '...",Ate chickpea curry on rice and salad. Drank 35...
559,2025-04-02T13:08:25-04:00,"{'item': 'water', 'action': 'drank', 'amount':...",Ate chickpea curry on rice and salad. Drank 35...
560,2025-04-02T15:13:45-04:00,"{'item': 'coffee', 'action': 'drank', 'amount'...",Drank coffee and 350ml water
560,2025-04-02T15:13:45-04:00,"{'item': 'water', 'action': 'drank', 'amount':...",Drank coffee and 350ml water
561,2025-04-02T17:09:25-04:00,"{'item': 'water', 'action': 'drank', 'amount':...",Drank 350ml water
562,2025-04-02T19:37:24-04:00,"{'size': 'small', 'effort': 'high', 'schema': ...",Small poop. High effort
563,2025-04-02T19:47:44-04:00,"{'item': 'water', 'action': 'drank', 'amount':...",Drank 500ml water


In [27]:
def get_type(x):
    schema = x.get("schema", None)
    if not schema:
        return "unknown"
    action = x.get("action", None)
    if not action:
        return schema
    return schema + "/" + action
events_df["type"] = events_df["data"].apply(get_type)
events_df["type"].value_counts()


consumed/drank        280
consumed/ate          225
consumed/took          98
pooped                 92
urinated               19
exercise/ran           12
peed                    5
exercise/meditated      4
meditated               2
mood                    2
symptom                 1
consumed/smoked         1
health                  1
event/ankied            1
event                   1
exercise/mediated       1
Name: type, dtype: int64

In [29]:
events_df.head()

,created_at,data,source_text,type
6,2025-02-14T02:39:12.090Z,"{'item': 'hot chocolate', 'action': 'drank', '...",had a hot chocolate,consumed/drank
15,2025-02-14T14:12:01.000Z,"{'effort': 'high', 'schema': 'pooped', 'poopTy...",Tried to poop but couldn’t,pooped
17,2025-02-14T16:03:48.000Z,"{'schema': 'symptom', 'startedAt': '2025-02-14...",Stomach feeling a bit unsettled,symptom
21,2025-02-14T16:18:03.000Z,"{'effort': 'low', 'schema': 'pooped', 'poopTyp...",Small poop. A little wet,pooped
22,2025-02-14T18:29:54.000Z,"{'item': 'vegetable and chickpea soup', 'actio...",Vegetable and chickpea soup and 1 glass of water,consumed/ate


In [34]:
water_df = events_df[events_df["type"] == "consumed/drank"]
food_df = events_df[events_df["type"] == "consumed/ate"]
supplements_df = events_df[events_df["type"] == "consumed/took"]
poops_df = events_df[events_df["type"] == "pooped"]

def data_to_cols(df):
    new_df = pd.concat([
        df[["created_at", "source_text"]].reset_index(drop=True),
        pd.DataFrame.from_records(df["data"].values).drop(["schema"], axis=1).reset_index(drop=True)
    ], axis=1)
    created_at = new_df.pop("created_at")
    if "startedAt" in new_df.columns:
      new_df.rename(columns={"startedAt": "started_at"}, inplace=True)
      new_df["started_at"] = pd.to_datetime(new_df["started_at"])
    else:
      new_df["started_at"] = pd.to_datetime(created_at)
    if "started_at" in new_df.columns:
        cols = ["started_at"] + [col for col in new_df.columns if col != "started_at"]
        new_df = new_df[cols]
    return new_df

poops_df = data_to_cols(poops_df)
water_df = data_to_cols(water_df)
food_df = data_to_cols(food_df)
supplements_df = data_to_cols(supplements_df)


print("water_df")
display(water_df.head())
print("food_df")
display(food_df.head())
print("supplements_df")
display(supplements_df.head())
print("poops_df")
display(poops_df.head())

water_df


,started_at,source_text,item,action,amount
0,2025-02-13 21:39:08-05:00,had a hot chocolate,hot chocolate,drank,1
1,2025-02-14 13:29:54-05:00,Vegetable and chickpea soup and 1 glass of water,water,drank,1 glass
2,2025-02-14 16:17:34-05:00,Drank 1L water,water,drank,1 liter
3,2025-02-15 08:30:00-05:00,"at 8:30am, at toast with peanut butter, a bana...",decaf tea,drank,1 cup
4,2025-02-14 10:30:53-05:00,30min ago had 2 egg omelette and half apple an...,tea,drank,1


food_df


,started_at,source_text,item,action,amount,healthiness,portionSize
0,2025-02-14 13:29:54-05:00,Vegetable and chickpea soup and 1 glass of water,vegetable and chickpea soup,ate,1,NaN,NaN
1,2025-02-15 08:30:00-05:00,"at 8:30am, at toast with peanut butter, a bana...",toast with peanut butter,ate,1,NaN,NaN
2,2025-02-15 08:30:00-05:00,"at 8:30am, at toast with peanut butter, a bana...",banana,ate,1,NaN,NaN
3,2025-02-14 10:30:53-05:00,30min ago had 2 egg omelette and half apple an...,egg omelette,ate,2,NaN,NaN
4,2025-02-14 10:30:53-05:00,30min ago had 2 egg omelette and half apple an...,apple,ate,0.5,NaN,NaN


supplements_df


,started_at,source_text,item,action,amount
0,2025-02-16 19:50:24-05:00,"Vitamin D, omega 3",Vitamin D,took,1 dose
1,2025-02-16 19:50:24-05:00,"Vitamin D, omega 3",omega 3,took,1 dose
2,2025-02-17 10:05:35-05:00,Vitamin d and omega. 30min ago,Vitamin D,took,1 dose
3,2025-02-17 10:05:35-05:00,Vitamin d and omega. 30min ago,omega 3,took,1 dose
4,2025-02-18 22:39:29-05:00,Took 1 magnesium,magnesium,took,1


poops_df


,started_at,source_text,effort,poopType,emptiness,empitness,size,duration
0,2025-02-14 09:12:01-05:00,Tried to poop but couldn’t,high,1,NaN,NaN,NaN,NaN
1,2025-02-14 11:18:03-05:00,Small poop. A little wet,low,6,NaN,NaN,NaN,NaN
2,2025-02-13 22:15:41-05:00,Tough poop. Long time. Small pieces,high,2,NaN,NaN,NaN,NaN
3,2025-02-14 22:18:16-05:00,Difficult poop. No success,high,1,NaN,NaN,NaN,NaN
4,2025-02-15 08:19:32-05:00,Long poop. Many pieces. Soft. Difficult,high,6,NaN,NaN,NaN,NaN


In [114]:
series = {}
series["poops"] = poops_df.set_index("started_at")["source_text"]

def format_food(x):
    items = []  
    for index, row in x.iterrows():
        if row["amount"] is not None and row["amount"] != "1":
            items.append(row["item"] + " (" + str(row["amount"]) + ")")
        else:
            items.append(row["item"])
    return ", ".join(items)
series["food"] = food_df.groupby("started_at").apply(format_food)
series["supplements"] = supplements_df.groupby("started_at").apply(lambda x: ", ".join(x["item"].str.lower().sort_values().values))
def format_water(x):
    items = []  
    for index, row in x.iterrows():
        amount = str(row["amount"] or "1 cup")
        amount = "1 cup" if amount == "1" else amount
        items.append(amount + " " + row["item"])
    return ", ".join(items)
series["water"] = water_df.groupby("started_at").apply(format_water)

In [121]:
import datetime

dfs = { key: s.reset_index() for key, s in series.items() }
dfs = {}
for key, s in series.items():
    df = s.reset_index()
    df.columns = ["started_at", "text"]
    dfs[key] = df
    # display(df.head())
df = pd.concat(dfs.values())
print(len(df))
df.sample(10)
df["date"] = df["started_at"].apply(lambda x: x.strftime("%Y-%m-%d"))
df.sort_values("started_at", inplace=True)
print(len(df))

res = '''# Food Log

Notes:
- nutty pudding has 20g of fiber
- super veggie has 10g of fiber
- each metamucil capsule has 1g of fiber

'''

for date, df in df.groupby("date"):
    formatted_date = datetime.datetime.strptime(date, "%Y-%m-%d").strftime("%A, %B %d, %Y")
    res += f"## {formatted_date}\n"
    for index, row in df.iterrows():
        time = row["started_at"].strftime("%I:%M%p")
        time = time.lstrip("0")
        res += f"- {time} {row['text']}\n"
    res += "\n"

# write to file
with open("log.md", "w") as f:
    f.write(res)


555
555



Example output:

```markdown
# Food Log

## March 21st 2025
- 10:30am vegetable soup, toast with peanut butter, 1 cup of water
- 12:30pm 1 cup water
- 1:30pm Tried to poop but couldn’t

## March 22nd 2025
- 11:30am 1 cup of water
- 12:30pm 1 cup of water
- 1:30pm 1 cup of water
```


